# IBKR News Fetch Smoke Test (`ib_async`)

This notebook only tests the `news_api` news-fetching path. It uses `ib_async`, checks IB news providers, optionally pulls one contract's historical news, and tests full-market BroadTape news subscriptions for the providers available to the account.

## 1. Setup

Restart the kernel before running if you previously used another IB connection in this notebook. Make sure TWS or IB Gateway API is enabled.

In [ ]:
from pathlib import Path
import logging
import sys
import time
from dataclasses import replace

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "news_api").exists() and (PROJECT_ROOT.parent / "news_api").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from ib_async import Contract, IB, Stock, util
except ModuleNotFoundError as exc:
    raise RuntimeError("This kernel does not have ib_async installed. Use the project's IB environment.") from exc

try:
    from ib_async.ib import StartupFetch
except Exception:
    StartupFetch = None

from news_api.cleaner import clean_headline, parse_headline_metadata, parse_ib_time
from news_api.config import SETTINGS
from news_api.models import NewsHeadline
from news_api.service import NewsService
from news_api.storage import SQLiteNewsStorage
from news_api.watchlist import normalize_watchlist

# ib_async's own logger often prints unicode escapes. Keep only our decoded errorEvent output.
for logger_name in ("ib_async", "ib_async.client", "ib_async.wrapper"):
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)

HOST = SETTINGS.host
PORT = SETTINGS.port
CLIENT_ID = 191

# Optional single-contract historical sanity check.
SYMBOL_TO_TEST = "ORCL"
HISTORICAL_RESULTS = 30

# Full-market BroadTape test. None means use every spec from SETTINGS.broadtape_specs.
BROADTAPE_SYMBOL = "ALL_NEWS"
BROADTAPE_PROVIDER_LIMIT = None
BROADTAPE_WAIT_SECONDS = 180
BROADTAPE_HISTORICAL_RESULTS = 10
BROADTAPE_SPECS = SETTINGS.broadtape_specs

PROVIDER_CODES = SETTINGS.provider_codes
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
DB_PATH = PROJECT_ROOT / "news_api" / "data" / f"news_fetch_smoke_test_{RUN_ID}.sqlite"

base_watchlist = normalize_watchlist()
if SYMBOL_TO_TEST not in base_watchlist:
    raise ValueError(f"{SYMBOL_TO_TEST} is not in news_api/watchlist.py")

print("project:", PROJECT_ROOT)
print("db:", DB_PATH)
print("host/port/client_id:", HOST, PORT, CLIENT_ID)
print("provider_codes:", PROVIDER_CODES)
print("broad tape symbol:", BROADTAPE_SYMBOL)

## 2. Test Service And SQLite

This uses a fresh smoke-test SQLite file and disables push/article fetch thresholds so the notebook focuses on fetching news headlines.

In [ ]:
class DryRunBarkClient:
    def push(self, analysis, priority):
        return "skipped", "news_fetch_smoke_test dry run"


test_settings = replace(
    SETTINGS,
    db_path=DB_PATH,
    article_fetch_score=999,
    default_push_score=999,
    portfolio_push_score=999,
)

storage = SQLiteNewsStorage(DB_PATH)
service_watchlist = {
    SYMBOL_TO_TEST: base_watchlist[SYMBOL_TO_TEST],
    BROADTAPE_SYMBOL: {
        "exchange": "NEWS",
        "priority": 1,
        "aliases": [BROADTAPE_SYMBOL, "BroadTape", "All News"],
    },
}
service = NewsService(
    settings=test_settings,
    watchlist=service_watchlist,
    storage=storage,
    bark_client=DryRunBarkClient(),
)
service.start()

def rows(sql, params=()):
    with storage.connect() as connection:
        return [dict(row) for row in connection.execute(sql, params).fetchall()]

def scalar(sql, params=()):
    with storage.connect() as connection:
        return connection.execute(sql, params).fetchone()[0]

print("service ready")

## 3. Connect IB

In [ ]:
def readable_ib_text(value):
    text = str(value)
    if "\\u" not in text and "\\x" not in text:
        return text
    try:
        return text.encode("utf-8").decode("unicode_escape")
    except UnicodeDecodeError:
        return text


util.startLoop()
ib = IB()
connect_kwargs = {"clientId": CLIENT_ID, "timeout": 20, "readonly": True}
if StartupFetch is not None:
    connect_kwargs["fetchFields"] = StartupFetch(0)

errors = []
def on_error(req_id, error_code, error_string, contract):
    message = readable_ib_text(error_string)
    row = {
        "time": time.strftime("%H:%M:%S"),
        "reqId": req_id,
        "errorCode": error_code,
        "errorString": message,
        "contract": readable_ib_text(contract) if contract else "",
    }
    errors.append(row)
    print(f"IBKR message [{row['time']}] reqId={req_id} code={error_code}: {message}")

ib.errorEvent += on_error
ib.connect(HOST, PORT, **connect_kwargs)
print("connected:", ib.isConnected())

## 4. News Providers

These are the provider codes the broad tape test will try. Some may still fail if the account lacks that specific real-time broad tape entitlement; the error output will show which one.

In [ ]:
providers = ib.reqNewsProviders()
provider_rows = [
    {"code": getattr(provider, "code", ""), "name": getattr(provider, "name", "")}
    for provider in providers
]
available_provider_set = {row["code"] for row in provider_rows if row.get("code")}
configured_provider_list = [code.strip() for code in PROVIDER_CODES.split("+") if code.strip()]
historical_provider_list = [
    code for code in configured_provider_list
    if not available_provider_set or code in available_provider_set
]
if not historical_provider_list:
    historical_provider_list = sorted(available_provider_set)
historical_provider_codes = "+".join(historical_provider_list)

print("provider count:", len(provider_rows))
print("historical provider codes:", historical_provider_codes)
try:
    import pandas as pd
    display(pd.DataFrame(provider_rows))
except ModuleNotFoundError:
    provider_rows


## 5. Optional Historical News Check For One Stock

In [ ]:
def make_stock_contract(symbol, item):
    primary_exchange = str(item.get("exchange", "SMART") or "SMART")
    kwargs = {}
    if primary_exchange.upper() != "SMART":
        kwargs["primaryExchange"] = primary_exchange
    return Stock(symbol, "SMART", "USD", **kwargs)


def ingest_historical_item(symbol, contract, item):
    headline_raw = readable_ib_text(getattr(item, "headline", "") or "")
    parsed = parse_headline_metadata(headline_raw)
    published_at, _local_time = parse_ib_time(str(getattr(item, "time", "") or ""))
    event = NewsHeadline(
        symbol=symbol,
        provider=str(getattr(item, "providerCode", "") or ""),
        article_id=str(getattr(item, "articleId", "") or ""),
        headline=clean_headline(headline_raw),
        headline_raw=headline_raw,
        publisher=parsed["publisher"],
        published_at=published_at,
        con_id=int(contract.conId or 0),
    )
    return service.ingest_headline(event)


stock_contract = make_stock_contract(SYMBOL_TO_TEST, base_watchlist[SYMBOL_TO_TEST])
qualified = ib.qualifyContracts(stock_contract)
if not qualified:
    raise RuntimeError(f"Could not qualify {SYMBOL_TO_TEST}: {stock_contract}")
stock_contract = qualified[0]
print(SYMBOL_TO_TEST, "conId=", stock_contract.conId, stock_contract)

before_raw = scalar("SELECT COUNT(*) FROM news_raw")
items = ib.reqHistoricalNews(stock_contract.conId, historical_provider_codes, "", "", HISTORICAL_RESULTS) or []
inserted = 0
for item in items:
    inserted += int(ingest_historical_item(SYMBOL_TO_TEST, stock_contract, item))
service.queue.join()
after_raw = scalar("SELECT COUNT(*) FROM news_raw")
print("historical returned:", len(items), "inserted:", inserted, "rows added:", after_raw - before_raw)

## 6. Full-Market BroadTape Real-Time News

This is the key full-coverage test. It subscribes NEWS contracts from `SETTINGS.broadtape_specs`, using IBKR BroadTape source codes such as `BRFG:BRFG_ALL@BRFG` instead of article provider codes such as `DJ-N`.


In [ ]:
def parse_broad_tape_specs(values):
    if isinstance(values, str):
        items = [item.strip() for item in values.split(",") if item.strip()]
    else:
        items = [str(item).strip() for item in values if str(item).strip()]

    specs = []
    for item in items:
        if "@" in item:
            symbol, exchange = item.rsplit("@", 1)
        else:
            symbol = item
            exchange = item.split(":", 1)[0]
        specs.append({
            "symbol": symbol.strip(),
            "exchange": exchange.strip(),
            "key": f"{symbol.strip()}@{exchange.strip()}",
        })
    return specs


def selected_broad_tape_specs():
    specs = parse_broad_tape_specs(BROADTAPE_SPECS)
    if BROADTAPE_PROVIDER_LIMIT is None:
        return specs
    return specs[:BROADTAPE_PROVIDER_LIMIT]


def make_broad_tape_contract(spec):
    contract = Contract()
    contract.symbol = spec["symbol"]
    contract.secType = "NEWS"
    contract.exchange = spec["exchange"]
    return contract


def qualify_broad_tape_contracts(specs):
    contracts = {}
    for spec in specs:
        contract = make_broad_tape_contract(spec)
        qualified = ib.qualifyContracts(contract)
        if not qualified or qualified[0] is None:
            print("skip unqualified broad tape:", spec["key"], contract)
            continue
        qualified_contract = qualified[0]
        # ib_async needs the conId from qualification for hashing, while IBKR
        # still expects the NEWS source code in exchange for reqMktData.
        qualified_contract.exchange = spec["exchange"]
        contracts[spec["key"]] = qualified_contract
    return contracts


if SETTINGS.market_data_type is not None:
    ib.reqMarketDataType(SETTINGS.market_data_type)
    print("market data type:", SETTINGS.market_data_type)

broad_specs = selected_broad_tape_specs()
broad_contracts = qualify_broad_tape_contracts(broad_specs)
print("broad tape provider count:", len(broad_contracts))
for key, contract in broad_contracts.items():
    print(key, contract)
if not broad_contracts:
    raise RuntimeError("No news provider available for BroadTape test.")


## 7. Full-Market BroadTape Historical News

This checks that the qualified all-news NEWS contracts can return historical headlines. It is useful even when the real-time window happens to be quiet.


In [ ]:
def ingest_broad_tape_historical_item(source_key, contract, item):
    headline_raw = readable_ib_text(getattr(item, "headline", "") or "")
    parsed = parse_headline_metadata(headline_raw)
    published_at, _local_time = parse_ib_time(str(getattr(item, "time", "") or ""))
    event = NewsHeadline(
        symbol=BROADTAPE_SYMBOL,
        provider=str(getattr(item, "providerCode", "") or source_key.rsplit("@", 1)[-1]),
        article_id=str(getattr(item, "articleId", "") or ""),
        headline=clean_headline(headline_raw),
        headline_raw=headline_raw,
        publisher=parsed["publisher"],
        published_at=published_at,
        con_id=int(getattr(contract, "conId", 0) or 0),
    )
    return service.ingest_headline(event)


before_broad_hist = scalar("SELECT COUNT(*) FROM news_raw")
broad_hist_returned = 0
broad_hist_inserted = 0
for source_key, contract in broad_contracts.items():
    provider_source = source_key.rsplit("@", 1)[-1]
    if not getattr(contract, "conId", 0):
        print("skip historical broad tape without conId:", source_key, contract)
        continue
    items = ib.reqHistoricalNews(
        int(contract.conId),
        provider_source,
        "",
        "",
        BROADTAPE_HISTORICAL_RESULTS,
    ) or []
    print("broad tape historical returned:", source_key, len(items))
    broad_hist_returned += len(items)
    for item in items:
        broad_hist_inserted += int(ingest_broad_tape_historical_item(source_key, contract, item))
service.queue.join()
after_broad_hist = scalar("SELECT COUNT(*) FROM news_raw")
print("broad tape historical returned total:", broad_hist_returned)
print("broad tape historical inserted:", broad_hist_inserted)
print("broad tape historical rows added:", after_broad_hist - before_broad_hist)


In [ ]:
def iter_news_ticks(ticker):
    for attr in ("newsTicks", "news", "tickNews"):
        value = getattr(ticker, attr, None)
        if isinstance(value, list):
            yield from value


def ingest_broad_tape_tick(provider_hint, tick):
    provider = str(getattr(tick, "providerCode", getattr(tick, "provider", "")) or provider_hint)
    article_id = str(getattr(tick, "articleId", "") or "")
    headline = readable_ib_text(getattr(tick, "headline", "") or "")
    timestamp = str(getattr(tick, "timeStamp", getattr(tick, "time", "")) or "")
    extra_data = readable_ib_text(getattr(tick, "extraData", "") or "")
    if not provider or not article_id or not headline:
        return False
    return service.ingest_tick_news(
        symbol=BROADTAPE_SYMBOL,
        timestamp=timestamp,
        provider=provider,
        article_id=article_id,
        headline=headline,
        ticker_id=None,
        extra_data=extra_data,
    )


NEWS_GENERIC_TICKS = "mdoff,292"
broad_tickers = {}
broad_seen = set()
before_broad = scalar("SELECT COUNT(*) FROM news_raw")

for provider_code, contract in broad_contracts.items():
    broad_tickers[provider_code] = ib.reqMktData(
        contract,
        genericTickList=NEWS_GENERIC_TICKS,
        snapshot=False,
        regulatorySnapshot=False,
    )
    print("subscribed broad tape:", provider_code, contract.symbol, NEWS_GENERIC_TICKS)

deadline = time.time() + BROADTAPE_WAIT_SECONDS
while time.time() < deadline:
    ib.sleep(10)
    observed = 0
    inserted = 0
    for provider_code, ticker in broad_tickers.items():
        for tick in iter_news_ticks(ticker):
            provider = str(getattr(tick, "providerCode", getattr(tick, "provider", "")) or provider_code)
            article_id = str(getattr(tick, "articleId", "") or "")
            key = (provider, article_id)
            if key in broad_seen:
                continue
            broad_seen.add(key)
            observed += 1
            inserted += int(ingest_broad_tape_tick(provider_code, tick))
    service.queue.join()
    current = scalar("SELECT COUNT(*) FROM news_raw")
    print(f"broad tape raw rows: {current} (+{current - before_broad}), observed={observed}, inserted={inserted}")
    if current > before_broad:
        break

after_broad = scalar("SELECT COUNT(*) FROM news_raw")
print("broad tape rows added:", after_broad - before_broad)
print("broad tape observed unique ticks:", len(broad_seen))

## 8. Inspect Stored News

In [ ]:
summary = rows("SELECT symbol, provider, COUNT(*) AS rows, MAX(received_at) AS latest_received_at FROM news_raw GROUP BY symbol, provider ORDER BY rows DESC, provider ASC")
latest = rows("SELECT symbol, published_at, provider, article_id, publisher, headline FROM news_raw ORDER BY received_at DESC LIMIT 30")
try:
    import pandas as pd
    display(pd.DataFrame(summary))
    display(pd.DataFrame(latest))
except ModuleNotFoundError:
    print(summary)
    print(latest)

## 9. Cleanup

In [ ]:
for provider_code, ticker in globals().get("broad_tickers", {}).items():
    try:
        ib.cancelMktData(ticker.contract)
    except Exception as exc:
        print("cancel broad tape MktData failed:", provider_code, exc)

service.stop()
if ib.isConnected():
    ib.disconnect()
print("stopped")